# 06. External Test 평가

**목적**: 학습·검증·내부 테스트에 사용되지 않은 외부 이미지로 모델 성능을 독립 평가합니다.

**입력**: `data/external_test/{refrigerator,washing_drying}/` — 서비스 대분류 기준 폴더

**모델**: `checkpoints/phase2_service_best_model.pth` (Phase 2 기준 모델)

**출력**: 서비스 대분류 및 세부 라벨 accuracy, confusion matrix, 오답 목록

> ⚠️ **이 노트북의 이미지는 절대 학습에 사용하지 않습니다.**  
> `data/external_test/` 의 이미지를 `data/processed/` 나 `data/split/` 으로 이동하지 마세요.

## 평가 구조

| 폴더 (Ground Truth) | 서비스 라벨 | 모델 예측 매핑 |
|---------------------|-------------|----------------|
| `refrigerator/` | refrigerator | refrigerator |
| `washing_drying/` | washing_drying | washer_dryer → washing_drying, wash_tower → washing_drying |

In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from PIL import Image as PILImage

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
import config

import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights

# ── 경로 설정 ────────────────────────────────────────────────────────────────────
CKPT_PATH      = os.path.join(config.CHECKPOINTS_DIR, 'phase2_service_best_model.pth')
EXT_TEST_DIR   = Path('data/external_test')
RESULT_CSV     = Path('data/metadata/external_test_predictions.csv')
IMG_SIZE       = 224

# 서비스 대분류 폴더명 (ground truth)
SERVICE_FOLDERS = ['refrigerator', 'washing_drying']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Checkpoint: {CKPT_PATH}')
print(f'External test dir: {EXT_TEST_DIR.resolve()}')

Device: cuda
Checkpoint: checkpoints\phase2_service_best_model.pth
External test dir: C:\Users\SSAFY\Desktop\miribom\data\external_test


In [2]:
# ── 체크포인트 로드 ───────────────────────────────────────────────────────────────
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
classes     = ckpt['classes']          # ['refrigerator', 'wash_tower', 'washer_dryer']
num_classes = len(classes)

model = efficientnet_v2_s(weights=None)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
model.load_state_dict(ckpt['model_state_dict'])
model = model.to(device)
model.eval()

print(f'Checkpoint phase : {ckpt["phase"]}  epoch : {ckpt["epoch"]}')
print(f'val_loss: {ckpt["val_loss"]:.4f}  val_acc: {ckpt["val_acc"]:.4f}')
print(f'세부 라벨 ({num_classes}개): {classes}')
print(f'\nSERVICE_LABEL_MAP: {config.SERVICE_LABEL_MAP}')

Checkpoint phase : phase2  epoch : 9
val_loss: 0.1029  val_acc: 0.9773
세부 라벨 (3개): ['refrigerator', 'wash_tower', 'washer_dryer']

SERVICE_LABEL_MAP: {'refrigerator': 'refrigerator', 'washer_dryer': 'washing_drying', 'wash_tower': 'washing_drying'}


In [3]:
# ── 이미지 스캔 ───────────────────────────────────────────────────────────────────
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

image_records = []  # [{path, service_true_label}]
for svc_label in SERVICE_FOLDERS:
    folder = EXT_TEST_DIR / svc_label
    if not folder.exists():
        print(f'  [없음] {folder}')
        continue
    files = sorted(f for f in folder.iterdir()
                   if f.suffix.lower() in ('.jpg', '.jpeg', '.png'))
    for f in files:
        image_records.append({'path': f, 'service_true_label': svc_label})
    print(f'  {svc_label}/: {len(files)}장')

total = len(image_records)
print(f'\n총 {total}장 발견')
if total == 0:
    print('\n⚠️  data/external_test/ 에 이미지가 없습니다.')
    print('   README.md 를 참고해 이미지를 배치한 후 다시 실행하세요.')

  refrigerator/: 74장
  washing_drying/: 70장

총 144장 발견


In [4]:
# ── 추론 ──────────────────────────────────────────────────────────────────────────
# 이미지가 없으면 이 셀을 건너뛰세요.
if total == 0:
    raise RuntimeError('이미지가 없습니다. 위 셀의 안내를 따르세요.')

results = []
with torch.no_grad():
    for rec in image_records:
        try:
            img = PILImage.open(rec['path']).convert('RGB')
            tensor = transform(img).unsqueeze(0).to(device)
            probs = torch.softmax(model(tensor), dim=1)[0]
            conf, pred_idx = probs.max(dim=0)
            fine_pred  = classes[pred_idx.item()]
            svc_pred   = config.SERVICE_LABEL_MAP[fine_pred]
            svc_true   = rec['service_true_label']
            results.append({
                'image_path':         str(rec['path']),
                'service_true_label': svc_true,
                'fine_pred_label':    fine_pred,
                'service_pred_label': svc_pred,
                'confidence':         round(conf.item(), 6),
                'service_correct':    svc_true == svc_pred,
                # 세부 라벨 기준 정답 여부는 ground truth 가 없으므로 N/A
                # (external_test 폴더는 서비스 라벨 기준)
            })
        except Exception as e:
            print(f'  [오류] {rec["path"].name}: {e}')

result_df = pd.DataFrame(results)
result_df.to_csv(RESULT_CSV, index=False, encoding='utf-8-sig')

n_correct = result_df['service_correct'].sum()
n_total   = len(result_df)
print(f'추론 완료: {n_total}장  |  정답 {n_correct}장  |  오답 {n_total - n_correct}장')
print(f'서비스 대분류 정확도: {n_correct/n_total:.1%}')
print(f'결과 저장: {RESULT_CSV}')
result_df.head()

추론 완료: 144장  |  정답 143장  |  오답 1장
서비스 대분류 정확도: 99.3%
결과 저장: data\metadata\external_test_predictions.csv


,image_path,service_true_label,fine_pred_label,service_pred_label,confidence,service_correct
0,data\external_test\refrigerator\ext_0000.jpg,refrigerator,refrigerator,refrigerator,0.960763,True
1,data\external_test\refrigerator\ext_0001.jpg,refrigerator,refrigerator,refrigerator,0.836896,True
2,data\external_test\refrigerator\ext_0002.jpg,refrigerator,refrigerator,refrigerator,0.996225,True
3,data\external_test\refrigerator\ext_0003.jpg,refrigerator,refrigerator,refrigerator,0.923031,True
4,data\external_test\refrigerator\ext_0005.jpg,refrigerator,refrigerator,refrigerator,0.999602,True


In [5]:
# ── 클래스별 정확도 + Confusion Matrix ────────────────────────────────────────────
svc_labels = SERVICE_FOLDERS
svc_idx    = {c: i for i, c in enumerate(svc_labels)}
n_svc      = len(svc_labels)

cm = np.zeros((n_svc, n_svc), dtype=int)
for _, row in result_df.iterrows():
    i = svc_idx.get(row['service_true_label'], -1)
    j = svc_idx.get(row['service_pred_label'], -1)
    if i >= 0 and j >= 0:
        cm[i][j] += 1

print('=== 서비스 대분류 클래스별 정확도 ===')
for i, cls in enumerate(svc_labels):
    tot = cm[i].sum()
    acc = cm[i][i] / tot if tot > 0 else 0
    print(f'  {cls}: {cm[i][i]}/{tot}  ({acc:.1%})')

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Greens)
plt.colorbar(im)
ax.set(xticks=range(n_svc), yticks=range(n_svc),
       xticklabels=svc_labels, yticklabels=svc_labels,
       ylabel='실제 (Ground Truth)', xlabel='예측',
       title=f'External Test — 서비스 대분류 CM ({n_total}장)')
max_v = cm.max() if cm.max() > 0 else 1
for i in range(n_svc):
    for j in range(n_svc):
        ax.text(j, i, cm[i][j], ha='center', va='center', fontsize=12,
                color='white' if cm[i][j] > max_v / 2 else 'black')
plt.tight_layout()
plt.show()

print()
print('=== 세부 라벨 예측 분포 (참고용) ===')
print(result_df.groupby(['service_true_label', 'fine_pred_label']).size().to_string())

=== 서비스 대분류 클래스별 정확도 ===
  refrigerator: 73/74  (98.6%)
  washing_drying: 70/70  (100.0%)

=== 세부 라벨 예측 분포 (참고용) ===
service_true_label  fine_pred_label
refrigerator        refrigerator       73
                    washer_dryer        1
washing_drying      wash_tower         26
                    washer_dryer       44


C:\Users\SSAFY\AppData\Local\Temp\ipykernel_5572\2551101892.py:31: UserWarning: Glyph 50696 (\N{HANGUL SYLLABLE YE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_5572\2551101892.py:31: UserWarning: Glyph 52769 (\N{HANGUL SYLLABLE CEUG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_5572\2551101892.py:31: UserWarning: Glyph 49892 (\N{HANGUL SYLLABLE SIL}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_5572\2551101892.py:31: UserWarning: Glyph 51228 (\N{HANGUL SYLLABLE JE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_5572\2551101892.py:31: UserWarning: Glyph 49436 (\N{HANGUL SYLLABLE SEO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_5572\2551101892.py:31: UserWarning: Glyph 48708 (\N{HANGUL SYLLABLE BI}) missing from font(s

In [6]:
# ── 오답 분석 ─────────────────────────────────────────────────────────────────────
wrong_df = result_df[~result_df['service_correct']].reset_index(drop=True)
print(f'서비스 대분류 오답: {len(wrong_df)}건')

if len(wrong_df) == 0:
    print('오답 없음 — 완벽한 분류!')
else:
    for _, row in wrong_df.iterrows():
        print(f'  {Path(row["image_path"]).name}: '
              f'true={row["service_true_label"]}  '
              f'pred={row["service_pred_label"]} ({row["fine_pred_label"]})  '
              f'conf={row["confidence"]:.3f}')

    n_show = min(5, len(wrong_df))
    fig, axes = plt.subplots(1, n_show, figsize=(4 * n_show, 4))
    if n_show == 1:
        axes = [axes]
    for ax_i, (_, row) in enumerate(wrong_df.head(n_show).iterrows()):
        img = PILImage.open(row['image_path']).convert('RGB')
        axes[ax_i].imshow(img)
        axes[ax_i].set_title(
            f'True: {row["service_true_label"]}\n'
            f'Pred: {row["fine_pred_label"]}\n'
            f'Conf: {row["confidence"]:.3f}',
            fontsize=9, color='red'
        )
        axes[ax_i].axis('off')
    plt.suptitle(f'서비스 오답 (상위 {n_show}건)', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

서비스 대분류 오답: 1건
  ext_0075.jpg: true=refrigerator  pred=washing_drying (washer_dryer)  conf=0.775


C:\Users\SSAFY\AppData\Local\Temp\ipykernel_5572\2676504286.py:29: UserWarning: Glyph 49436 (\N{HANGUL SYLLABLE SEO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_5572\2676504286.py:29: UserWarning: Glyph 48708 (\N{HANGUL SYLLABLE BI}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_5572\2676504286.py:29: UserWarning: Glyph 49828 (\N{HANGUL SYLLABLE SEU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_5572\2676504286.py:29: UserWarning: Glyph 50724 (\N{HANGUL SYLLABLE O}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_5572\2676504286.py:29: UserWarning: Glyph 45813 (\N{HANGUL SYLLABLE DAB}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_5572\2676504286.py:29: UserWarning: Glyph 49345 (\N{HANGUL SYLLABLE SANG}) missing from font(s

In [7]:
# ── Low Confidence 정답 분석 ─────────────────────────────────────────────────────
CONF_THRESHOLD = 0.85
low_conf = (
    result_df[result_df['service_correct']]
    .sort_values('confidence', ascending=True)
    .reset_index(drop=True)
)

very_low = low_conf[low_conf['confidence'] < CONF_THRESHOLD]
print(f'정답 중 confidence < {CONF_THRESHOLD}: {len(very_low)}건')
if len(very_low) > 0:
    for _, row in very_low.iterrows():
        print(f'  {Path(row["image_path"]).name}: '
              f'svc={row["service_true_label"]}  '
              f'pred_fine={row["fine_pred_label"]}  '
              f'conf={row["confidence"]:.3f}')

print()
print('=== 신뢰도 분포 ===')
bins = [0, 0.5, 0.7, 0.85, 0.95, 1.01]
labels = ['<0.50', '0.50-0.70', '0.70-0.85', '0.85-0.95', '≥0.95']
result_df['conf_bin'] = pd.cut(result_df['confidence'], bins=bins, labels=labels, right=False)
dist = result_df.groupby('conf_bin', observed=True)['service_correct'].agg(['count', 'sum'])
dist.columns = ['total', 'correct']
dist['acc'] = (dist['correct'] / dist['total']).map(lambda x: f'{x:.1%}' if dist['total'].sum() > 0 else 'N/A')
print(dist.to_string())

정답 중 confidence < 0.85: 36건
  ext_0045.jpg: svc=washing_drying  pred_fine=wash_tower  conf=0.397
  ext_0049.jpg: svc=washing_drying  pred_fine=washer_dryer  conf=0.444
  ext_0016.jpg: svc=refrigerator  pred_fine=refrigerator  conf=0.450
  ext_0003.jpg: svc=washing_drying  pred_fine=washer_dryer  conf=0.503
  ext_0056.jpg: svc=refrigerator  pred_fine=refrigerator  conf=0.543
  ext_0073.jpg: svc=washing_drying  pred_fine=washer_dryer  conf=0.562
  ext_0052.jpg: svc=washing_drying  pred_fine=washer_dryer  conf=0.572
  ext_0016.jpg: svc=washing_drying  pred_fine=washer_dryer  conf=0.576
  ext_0009.jpg: svc=washing_drying  pred_fine=wash_tower  conf=0.584
  ext_0059.jpg: svc=washing_drying  pred_fine=washer_dryer  conf=0.585
  ext_0010.jpg: svc=washing_drying  pred_fine=wash_tower  conf=0.610
  ext_0060.jpg: svc=washing_drying  pred_fine=wash_tower  conf=0.620
  ext_0033.jpg: svc=washing_drying  pred_fine=washer_dryer  conf=0.623
  ext_0015.jpg: svc=washing_drying  pred_fine=washer_dryer  c

In [8]:
# ── 세부 라벨 예측 분포 시각화 (washing_drying 내부) ─────────────────────────────
# washing_drying 폴더의 이미지가 wash_tower vs washer_dryer 중 어느 쪽으로 예측됐는지 확인
wd_subset = result_df[result_df['service_true_label'] == 'washing_drying'].copy()
if len(wd_subset) > 0:
    dist = wd_subset['fine_pred_label'].value_counts()
    print('washing_drying 폴더 → 세부 라벨 예측 분포:')
    for lbl, cnt in dist.items():
        pct = cnt / len(wd_subset) * 100
        print(f'  {lbl}: {cnt}장  ({pct:.1f}%)')
    print()
    print('(참고: washer_dryer / wash_tower 는 서비스에서 동일하게 washing_drying으로 처리됩니다.)')
else:
    print('washing_drying/ 에 이미지가 없습니다.')

washing_drying 폴더 → 세부 라벨 예측 분포:
  washer_dryer: 44장  (62.9%)
  wash_tower: 26장  (37.1%)

(참고: washer_dryer / wash_tower 는 서비스에서 동일하게 washing_drying으로 처리됩니다.)
